# 08 · LBD-Q-SAT Evaluation

## Colab setup

In [ ]:
import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/neurosat"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, PROJECT_DIR)

print("Working folder:", os.getcwd())
print("Files:", os.listdir())

## Imports

In [ ]:
import pickle
import random
import time

import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

from cdcl import solve, vsids_literal
from qnet import QNetwork, q_literal

## Settings

In [ ]:
SEED = 42
DIM = 64
ROUNDS = 4
MAX_DECISIONS = 5000
GEN_VARS = 80
GEN_SIZE = 50
GLUE_LBD = 2

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Load data

In [ ]:
with open("rl_data.pkl", "rb") as f:
    rl_data = pickle.load(f)

with open("baselines.pkl", "rb") as f:
    baselines = pickle.load(f)

with open("test_large.pkl", "rb") as f:
    large_test_data = pickle.load(f)

eval_sets = rl_data["eval"]
gen_data = large_test_data[GEN_VARS][:GEN_SIZE]

for name, data in eval_sets.items():
    print(f"{name:6s}: {len(data)} formulas")
print(f"{GEN_VARS:<6d}: {len(gen_data)} formulas (generalization)")

## Load trained agents

In [ ]:
agents = {}
for mode, name in [("flat", "Q flat"), ("lbd", "Q LBD")]:
    model = QNetwork(DIM, ROUNDS).to(device)
    model.load_state_dict(torch.load(f"qnet_{mode}.pt", map_location=device))
    model.eval()
    agents[name] = model

print("Loaded:", list(agents.keys()))

## Run function

In [ ]:
def run(problem, choose):
    start = time.perf_counter()
    solver = solve(problem["n_vars"], problem["clauses"], choose, MAX_DECISIONS)
    elapsed = time.perf_counter() - start

    return {
        "n_vars": problem["n_vars"],
        "label": problem["label"],
        "finished": solver.result is not None,
        "correct": solver.result == bool(problem["label"]),
        "decisions": solver.decisions,
        "conflicts": solver.conflicts,
        "learned": len(solver.lbds),
        "mean_lbd": sum(solver.lbds) / len(solver.lbds) if solver.lbds else 0,
        "lbds": solver.lbds,
        "ms": elapsed * 1000,
    }

## Run agents on evaluation sets

In [ ]:
rows = []

for set_name, data in eval_sets.items():
    for method, model in agents.items():
        choose = q_literal(model, device)
        for i, problem in enumerate(data):
            row = run(problem, choose)
            row.update({"set": set_name, "method": method, "formula": i})
            rows.append(row)
        print(f"Done {set_name} with {method}")

agent_results = pd.DataFrame(rows)

## Baseline results

In [ ]:
baseline_results = baselines["results"].rename(columns={"heuristic": "method"}).copy()
baseline_results["formula"] = baseline_results.groupby(["set", "method"]).cumcount()

print(baseline_results.groupby(["set", "method"]).size())

## Generalization to larger formulas

In [ ]:
gen_methods = {"VSIDS": vsids_literal}
for method, model in agents.items():
    gen_methods[method] = q_literal(model, device)

rows = []
for method, choose in gen_methods.items():
    for i, problem in enumerate(gen_data):
        row = run(problem, choose)
        row.update({"set": str(GEN_VARS), "method": method, "formula": i})
        rows.append(row)
    print(f"Done {GEN_VARS} with {method}")

gen_results = pd.DataFrame(rows)

## Combine results

In [ ]:
results = pd.concat([baseline_results, agent_results, gen_results], ignore_index=True)

SET_ORDER = list(eval_sets.keys()) + [str(GEN_VARS)]
METHOD_ORDER = ["VSIDS", "DLIS", "Random", "Q flat", "Q LBD"]

print("Rows:", len(results))
print(results.groupby(["set", "method"]).size().unstack())

## Correctness check

In [ ]:
finished = results[results["finished"]]

print("Unfinished runs:")
print(results[~results["finished"]].groupby(["set", "method"]).size())
print()
print("Wrong finished answers:", (~finished["correct"]).sum())

## Mean decisions

In [ ]:
mean_decisions = results.pivot_table(index="set", columns="method", values="decisions", aggfunc="mean")
mean_decisions = mean_decisions.reindex(index=SET_ORDER, columns=METHOD_ORDER)
print(mean_decisions.round(1))

## Median decisions

In [ ]:
median_decisions = results.pivot_table(index="set", columns="method", values="decisions", aggfunc="median")
median_decisions = median_decisions.reindex(index=SET_ORDER, columns=METHOD_ORDER)
print(median_decisions)

## Decisions relative to VSIDS

In [ ]:
relative = median_decisions.div(median_decisions["VSIDS"], axis=0)
print(relative.round(2))

## SAT and UNSAT separately

In [ ]:
by_label = results.pivot_table(index=["set", "label"], columns="method", values="decisions", aggfunc="median")
by_label = by_label.reindex(columns=METHOD_ORDER)
print(by_label)

## Learned clause quality

In [ ]:
def clause_stats(group):
    lbds = [lbd for row in group["lbds"] for lbd in row]
    if not lbds:
        return pd.Series({"learned": 0, "mean_lbd": 0.0, "glue_share": 0.0})
    return pd.Series({
        "learned": len(lbds),
        "mean_lbd": sum(lbds) / len(lbds),
        "glue_share": sum(lbd <= GLUE_LBD for lbd in lbds) / len(lbds),
    })


quality = results.groupby(["set", "method"])[["lbds"]].apply(clause_stats)
print(quality.round(3))

## Decisions plot

In [ ]:
median_decisions.plot(kind="bar", figsize=(10, 4), logy=True)
plt.xlabel("Variables")
plt.ylabel("Median decisions")
plt.xticks(rotation=0)
plt.legend(title="Method")
plt.tight_layout()
plt.savefig("final_decisions.png", dpi=150)
plt.show()

## LBD distribution of the two agents

In [ ]:
fig, axes = plt.subplots(1, len(SET_ORDER), figsize=(4 * len(SET_ORDER), 3.5), sharey=True)

for ax, set_name in zip(axes, SET_ORDER):
    for method in ["VSIDS", "Q flat", "Q LBD"]:
        subset = results[(results["set"] == set_name) & (results["method"] == method)]
        lbds = [lbd for row in subset["lbds"] for lbd in row]
        if lbds:
            ax.hist(lbds, bins=range(1, 21), alpha=0.5, density=True, label=method)
    ax.set_title(f"n = {set_name}")
    ax.set_xlabel("LBD")

axes[0].set_ylabel("Share of learned clauses")
axes[0].legend()
plt.tight_layout()
plt.savefig("final_lbd_distribution.png", dpi=150)
plt.show()

## Flat vs LBD per formula

In [ ]:
fig, axes = plt.subplots(1, len(SET_ORDER), figsize=(4 * len(SET_ORDER), 4))

for ax, set_name in zip(axes, SET_ORDER):
    flat = results[(results["set"] == set_name) & (results["method"] == "Q flat")].sort_values("formula")["decisions"].values
    lbd = results[(results["set"] == set_name) & (results["method"] == "Q LBD")].sort_values("formula")["decisions"].values
    ax.scatter(flat + 1, lbd + 1, s=10, alpha=0.5)
    top = max(flat.max(), lbd.max()) + 1
    ax.plot([1, top], [1, top], color="gray", linestyle="--")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"n = {set_name}")
    ax.set_xlabel("Decisions, flat reward")

axes[0].set_ylabel("Decisions, LBD reward")
plt.tight_layout()
plt.savefig("final_flat_vs_lbd.png", dpi=150)
plt.show()

## Win rates

In [ ]:
def paired(set_name, method):
    subset = results[(results["set"] == set_name) & (results["method"] == method)]
    return subset.sort_values("formula")["decisions"].values


print(f"{'n':>6s} | {'comparison':>18s} | {'better':>6s} | {'worse':>6s} | {'tie':>6s}")
for set_name in SET_ORDER:
    lbd = paired(set_name, "Q LBD")
    for other in ["Q flat", "VSIDS"]:
        o = paired(set_name, other)
        print(f"{set_name:>6s} | {'Q LBD vs ' + other:>18s} | {(lbd < o).mean():6.2f} | {(lbd > o).mean():6.2f} | {(lbd == o).mean():6.2f}")

## Significance test

In [ ]:
for set_name in SET_ORDER:
    flat = paired(set_name, "Q flat")
    lbd = paired(set_name, "Q LBD")
    if (flat != lbd).any():
        statistic, p_value = wilcoxon(flat, lbd)
        print(f"n={set_name:6s} | Wilcoxon p-value, flat vs LBD: {p_value:.4f}")
    else:
        print(f"n={set_name:6s} | identical decisions, no test")

## Solving time

In [ ]:
mean_time = results.pivot_table(index="set", columns="method", values="ms", aggfunc="mean")
mean_time = mean_time.reindex(index=SET_ORDER, columns=METHOD_ORDER)
print(mean_time.round(1))

## Final summary

In [ ]:
summary = pd.DataFrame({
    "VSIDS median decisions": median_decisions["VSIDS"],
    "Q flat median decisions": median_decisions["Q flat"],
    "Q LBD median decisions": median_decisions["Q LBD"],
    "Q flat mean LBD": [quality.loc[(s, "Q flat"), "mean_lbd"] for s in SET_ORDER],
    "Q LBD mean LBD": [quality.loc[(s, "Q LBD"), "mean_lbd"] for s in SET_ORDER],
}, index=SET_ORDER)

print(summary.round(2))

## Save

In [ ]:
with open("final_results.pkl", "wb") as f:
    pickle.dump({"results": results, "summary": summary, "quality": quality}, f)

print("Saved final_results.pkl and the final plots")